# PySWEB example: run SSEBop from a notebook with `import pysweb`

This notebook shows the package-backed workflow for preparing Landsat + ERA5-Land inputs and running SSEBop directly from Python. It also includes an optional SWB section for the final soil-water-balance run once SWB preprocess outputs already exist.

## Configure paths and run settings

This is the notebook equivalent of the old shell flow. Update the paths below before running the cells. `RUN_PREPARE_INPUTS`, `RUN_SSEBOP`, and `RUN_SWB` default to `False` so the notebook is safe to open and inspect before launching downloads or model runs.

In [ ]:
from pathlib import Path

PROJECT_DIR = Path("/g/data/ym05/github/PySWEB")
RUN_ROOT = Path("/g/data/ym05/sweb_model/notebook_runs")
RUN_NAME = "Llara"
DATE_RANGE = "2024-01-01 to 2024-01-31"
EXTENT = [147.20, -35.10, 147.30, -35.00]

START_DATE, END_DATE = [part.strip() for part in DATE_RANGE.split("to")]
START_TAG = START_DATE.replace("-", "")
END_TAG = END_DATE.replace("-", "")

RUN_DIR = RUN_ROOT / RUN_NAME
LANDSAT_DIR = RUN_DIR / "inputs" / "landsat"
MET_RAW_DIR = RUN_DIR / "inputs" / "met" / "era5land" / "raw"
MET_STACK_DIR = RUN_DIR / "inputs" / "met" / "era5land" / "stack"
SSEBOP_OUTPUT_DIR = RUN_DIR / "outputs" / "ssebop"
SWB_PREPROCESS_DIR = RUN_DIR / "outputs" / "sweb_preprocess"
SWB_OUTPUT_DIR = RUN_DIR / "outputs" / "sweb"

GEE_CONFIG = PROJECT_DIR / "gee_config" / "base_gee.yaml"
DEM = Path("/path/to/dem.tif")

RUN_PREPARE_INPUTS = False
RUN_SSEBOP = False
RUN_SWB = False

for path in [LANDSAT_DIR, MET_RAW_DIR, MET_STACK_DIR, SSEBOP_OUTPUT_DIR, SWB_OUTPUT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print(f"Notebook run directory: {RUN_DIR}")
print(f"GEE config: {GEE_CONFIG}")
print(f"DEM: {DEM}")

In [ ]:
import sys

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import pysweb

if RUN_PREPARE_INPUTS:
    pysweb.ssebop.prepare_inputs(
        date_range = DATE_RANGE,
        extent = EXTENT,
        met_source = "era5land",
        landsat_dir = str(LANDSAT_DIR),
        met_raw_dir = str(MET_RAW_DIR),
        met_stack_dir = str(MET_STACK_DIR),
        dem = str(DEM),
        gee_config = str(GEE_CONFIG),
    )
else:
    print("Set RUN_PREPARE_INPUTS = True to download Landsat and ERA5-Land inputs.")

In [ ]:
if RUN_SSEBOP:
    pysweb.ssebop.run(
        date_range = DATE_RANGE,
        landsat_dir = str(LANDSAT_DIR),
        met_dir = str(MET_STACK_DIR),
        dem = str(DEM),
        output_dir = str(SSEBOP_OUTPUT_DIR),
        workers = 4,
    )
else:
    print("Set RUN_SSEBOP = True to execute the package-backed SSEBop workflow.")

In [ ]:
import matplotlib.pyplot as plt
import xarray as xr
from IPython.display import display

ssebop_nc = SSEBOP_OUTPUT_DIR / f"et_daily_ssebop_{START_DATE}_{END_DATE}.nc"

if ssebop_nc.exists():
    ds_ssebop = xr.open_dataset(ssebop_nc)
    display(ds_ssebop)
    ds_ssebop["ET"].isel(time=0).plot(cmap="viridis")
    plt.title(f"SSEBop ET on {str(ds_ssebop.time.values[0])[:10]}")
    plt.show()
else:
    print(f"Run SSEBop first or update the path: {ssebop_nc}")

## Optional SWB final run

`pysweb.swb.run(...)` is package-backed, but SWB preprocess and calibration are still driven by `workflows/3_sweb_preprocess_inputs.py` and `workflows/4_sweb_calib_domain.py`. Use the cells below only after those forcing and soil inputs already exist.

In [ ]:
rain_nc = SWB_PREPROCESS_DIR / f"rain_daily_{START_TAG}_{END_TAG}.nc"
effective_precip_nc = SWB_PREPROCESS_DIR / f"effective_precip_daily_{START_TAG}_{END_TAG}.nc"
et_nc = SWB_PREPROCESS_DIR / f"et_daily_{START_TAG}_{END_TAG}.nc"
t_nc = SWB_PREPROCESS_DIR / f"t_daily_{START_TAG}_{END_TAG}.nc"

if RUN_SWB:
    pysweb.swb.run(
        precip = str(rain_nc),
        effective_precip = str(effective_precip_nc),
        et = str(et_nc),
        t = str(t_nc),
        soil_dir = str(SWB_PREPROCESS_DIR),
        output_dir = str(SWB_OUTPUT_DIR),
        start_date = START_DATE,
        end_date = END_DATE,
        workers = 4,
    )
else:
    print("Set RUN_SWB = True after SWB preprocess/calibration inputs are ready.")
    print(f"Expected SWB preprocess directory: {SWB_PREPROCESS_DIR}")

In [ ]:
swb_outputs = sorted(SWB_OUTPUT_DIR.glob("SWEB_RZSM*.nc"))

if swb_outputs:
    ds_swb = xr.open_dataset(swb_outputs[-1])
    display(ds_swb)
    preview_var = "profile_sm" if "profile_sm" in ds_swb.data_vars else list(ds_swb.data_vars)[0]
    ds_swb[preview_var].isel(time=0).plot(cmap="Blues")
    plt.title(f"{preview_var} on {str(ds_swb.time.values[0])[:10]}")
    plt.show()
else:
    print(f"No SWB output found in {SWB_OUTPUT_DIR}")